Advanced Prompting with Mistral

**Improvements:**
- Chain-of-Thought prompting
- Similar example selection
- Better reasoning


## 1. Setup

In [ ]:
!pip install -q transformers accelerate bitsandbytes
from google.colab import drive
#drive.mount('/content/drive')
import json, re, numpy as np, torch, random
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from tqdm import tqdm
random.seed(42)

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, load_in_4bit=True, torch_dtype=torch.bfloat16, device_map='auto')
pipe = pipeline('text-generation', model=model, tokenizer=tokenizer, max_new_tokens=50, temperature=0.1)
print('✓ Mistral loaded')

In [ ]:
def load_jsonl(fp):
    return [json.loads(l) for l in open(fp) if l.strip()]

train_rest = load_jsonl('/content/eng_restaurant_small_train.jsonl')
train_laptop = load_jsonl('/content/eng_laptop_small_train.jsonl')
valid_rest = load_jsonl('/content/eng_restaurant_valid.jsonl')
valid_laptop = load_jsonl('/content/eng_laptop_valid.jsonl')
test_rest = load_jsonl('/content/eng_restaurant_dev_task1.jsonl')
test_laptop = load_jsonl('/content/eng_laptop_dev_task1.jsonl')

def extract_pairs(data, domain):
    pairs = []
    for item in data:
        for quad in item['Quadruplet']:
            aspect = quad['Aspect'] if quad['Aspect'] != 'NULL' else quad['Category']
            v, a = map(float, quad['VA'].split('#'))
            pairs.append({'text': item['Text'], 'aspect': aspect, 'v': v, 'a': a, 'domain': domain})
    return pairs

train_rest_pairs = extract_pairs(train_rest, 'restaurant')
train_laptop_pairs = extract_pairs(train_laptop, 'laptop')
print(f'Train: {len(train_rest_pairs)+len(train_laptop_pairs)}')

## 2. Advanced CoT Prompting

In [ ]:
def select_similar(text, aspect, pairs, k=5):
    same = [p for p in pairs if p['aspect'].lower() == aspect.lower()]
    if len(same) >= k: return random.sample(same, k)
    others = [p for p in pairs if p['aspect'].lower() != aspect.lower()]
    return same + random.sample(others, k - len(same))

def create_cot_prompt(text, aspect, domain, examples):
    p = f'''Analyze {domain} sentiment step-by-step:
1. Identify sentiment words
2. Valence (1-9: negative to positive)
3. Arousal (1-9: calm to excited)\n\n'''
    for ex in examples:
        p += f'Text: "{ex["text"]}"\nAspect: {ex["aspect"]}\nAnalysis: '
        p += ('Positive, ' if ex['v'] > 6 else 'Negative, ' if ex['v'] < 4 else 'Neutral, ')
        p += ('high intensity' if ex['a'] > 6 else 'moderate')
        p += f'\nVA: {ex["v"]:.2f}#{ex["a"]:.2f}\n\n'
    p += f'Text: "{text}"\nAspect: {aspect}\nAnalysis: '
    return p

def parse_va(text):
    m = re.search(r'(\d+\.\d{2})#(\d+\.\d{2})', text)
    return (max(1.0, min(9.0, float(m.group(1)))), max(1.0, min(9.0, float(m.group(2))))) if m else (5.0, 5.0)

def predict_va(text, aspect, domain, pairs):
    examples = select_similar(text, aspect, pairs)
    prompt = create_cot_prompt(text, aspect, domain, examples)
    output = pipe(prompt)
    return parse_va(output[0]['generated_text'][len(prompt):])

## 3. Validate

In [ ]:
def evaluate_rmse(data, domain, pairs):
    errors = []
    for item in tqdm(data, desc=domain):
        for quad in item['Quadruplet']:
            aspect = quad['Aspect'] if quad['Aspect'] != 'NULL' else quad['Category']
            gold_v, gold_a = map(float, quad['VA'].split('#'))
            pred_v, pred_a = predict_va(item['Text'], aspect, domain, pairs)
            errors.append((pred_v - gold_v)**2 + (pred_a - gold_a)**2)
    return np.sqrt(np.mean(errors))

print('Validation RMSE (20 instances):')
rest_rmse = evaluate_rmse(valid_rest[:20], 'restaurant', train_rest_pairs)
laptop_rmse = evaluate_rmse(valid_laptop[:20], 'laptop', train_laptop_pairs)
print(f'Restaurant: {rest_rmse:.3f}\nLaptop: {laptop_rmse:.3f}\nAverage: {(rest_rmse+laptop_rmse)/2:.3f}')

## 4. Final Model & Test Predictions

In [ ]:
full_rest = extract_pairs(train_rest + valid_rest, 'restaurant')
full_laptop = extract_pairs(train_laptop + valid_laptop, 'laptop')

def predict_test(test_data, domain, pairs):
    preds = []
    for item in tqdm(test_data, desc=domain):
        av_list = [{'Aspect': a, 'VA': f'{predict_va(item["Text"], a, domain, pairs)[0]:.2f}#{predict_va(item["Text"], a, domain, pairs)[1]:.2f}'} for a in item['Aspect']]
        preds.append({'ID': item['ID'], 'Aspect_VA': av_list})
    return preds

rest_preds = predict_test(test_rest, 'restaurant', full_rest)
laptop_preds = predict_test(test_laptop, 'laptop', full_laptop)

def save(preds, path):
    with open(path, 'w') as f:
        for p in preds: f.write(json.dumps(p) + '\n')
save(rest_preds, '/content/pred_eng_restaurant.jsonl')
save(laptop_preds, '/content/pred_eng_laptop.jsonl')

from google.colab import files
files.download('/content/pred_eng_restaurant.jsonl')
files.download('/content/pred_eng_laptop.jsonl')